# Learn and Help — Python DS
### [www.learnandhelp.com](https://www.learnandhelp.com)

---

# Week 26: Web Scraping with Beautiful Soup
**Python for Data Science (Python DS)** — Learn and Help

---

In this notebook you will learn to:
- Parse HTML using `BeautifulSoup`
- Find tags with `.find()` and `.find_all()`
- Extract text and attributes from HTML elements
- Scrape an HTML table into a Pandas DataFrame
- Apply ethical scraping practices

> **Note:** All examples in Sections 1–5 use local HTML strings so you can run them instantly in Colab without needing internet access. Section 6 shows a live web request.

## Setup — Install and Import Libraries

In [ ]:
# Install Beautiful Soup (already available in Colab, but good practice)
!pip install beautifulsoup4 --quiet

# Imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

print('✅ All libraries imported successfully!')

---
## Section 1 — Parsing HTML: Your First BeautifulSoup Object

A `BeautifulSoup` object is like a **searchable map** of an HTML document.  
You create one by passing raw HTML text and a parser name.

In [ ]:
# A simple HTML page stored as a Python string
html = """
<!DOCTYPE html>
<html>
  <head><title>My Sports Page</title></head>
  <body>
    <h1>Top NBA Players 2024</h1>
    <p class="intro">Here are this season's scoring leaders.</p>
    <p>Data is updated weekly.</p>
  </body>
</html>
"""

# Parse the HTML
soup = BeautifulSoup(html, 'html.parser')

# See the nicely formatted version
print(soup.prettify())

In [ ]:
# Access tags directly like attributes on the soup object
print('Title tag:  ', soup.title)          # <title>My Sports Page</title>
print('Title text: ', soup.title.text)     # My Sports Page
print()
print('H1 tag:     ', soup.h1)             # <h1>Top NBA Players 2024</h1>
print('H1 text:    ', soup.h1.text)        # Top NBA Players 2024
print()
print('First <p>:  ', soup.p)             # first <p> tag only
print('First <p> text:', soup.p.text)

---
## Section 2 — `.find()` and `.find_all()`

| Method | Returns | Use when... |
|---|---|---|
| `.find()` | The **first** matching tag | You want one specific element |
| `.find_all()` | A **list** of all matching tags | You want multiple elements |

In [ ]:
html2 = """
<html><body>
  <h2 id="section-a">Basketball</h2>
  <p class="intro">The NBA season runs October through June.</p>
  <p class="detail">30 teams compete for the championship.</p>
  <h2 id="section-b">Baseball</h2>
  <p class="intro">The MLB season runs March through October.</p>
  <p class="detail">30 teams play 162 games each.</p>
</body></html>
"""

soup2 = BeautifulSoup(html2, 'html.parser')

# --- .find() examples ---
print('=== .find() — returns the FIRST match ===')
print(soup2.find('h2'))                         # first <h2>
print(soup2.find('h2').text)                    # its text
print(soup2.find('p', class_='intro').text)     # first <p class="intro">
print(soup2.find('h2', id='section-b').text)    # <h2 id="section-b">

In [ ]:
# --- .find_all() examples ---
print('=== .find_all() — returns ALL matches ===')

all_h2 = soup2.find_all('h2')
print(f'Found {len(all_h2)} <h2> tags:')
for tag in all_h2:
    print('  -', tag.text)

print()

all_intros = soup2.find_all('p', class_='intro')
print(f'Found {len(all_intros)} <p class="intro"> tags:')
for tag in all_intros:
    print('  -', tag.text)

---
## Section 3 — Extracting Text and Attributes

Once you find a tag, you need to pull out its **content** (text) or its **attributes** (href, class, src, etc.).

In [ ]:
html3 = """
<html><body>
  <a href="https://www.nba.com" class="league-link" data-sport="basketball">NBA Official Site</a>
  <a href="https://www.mlb.com" class="league-link" data-sport="baseball">MLB Official Site</a>
  <a href="https://www.nfl.com" class="league-link" data-sport="football">NFL Official Site</a>
  <img src="nba_logo.png" alt="NBA Logo" width="100">
</body></html>
"""

soup3 = BeautifulSoup(html3, 'html.parser')

# Getting text
link = soup3.find('a')
print('Text using .text:           ', link.text)
print('Text using .get_text():     ', link.get_text())
print('Text stripped of whitespace:', link.get_text(strip=True))

In [ ]:
# Getting attribute values
link = soup3.find('a')

# Method 1: dictionary-style (crashes if attribute is missing!)
print('href (dict style):  ', link['href'])

# Method 2: .get() style — SAFER, returns None if attribute is missing
print('href (.get style):  ', link.get('href'))
print('class attribute:    ', link.get('class'))        # returns a list!
print('data-sport:         ', link.get('data-sport'))
print('missing attribute:  ', link.get('id'))           # None — no crash

In [ ]:
# Scraping all links from a page — a very common pattern
print('All links on the page:')
print(f'{"Link Text":<25} {"URL"}')
print('-' * 60)

for a in soup3.find_all('a'):
    text = a.get_text(strip=True)
    url  = a.get('href', 'No href')
    sport = a.get('data-sport', 'unknown')
    print(f'{text:<25} {url}  [{sport}]')

---
## Section 4 — Scraping an HTML Table → Pandas DataFrame

This is the most important skill for data science scraping!  
The pattern is always the same: **find the table → extract headers → loop through rows → build DataFrame**.

In [ ]:
# HTML table with NBA player stats
html4 = """
<html><body>
<h2>NBA Scoring Leaders 2024–25</h2>
<table id="players" class="stats-table">
  <tr>
    <th>Rank</th><th>Player</th><th>Team</th><th>PPG</th><th>RPG</th><th>APG</th>
  </tr>
  <tr>
    <td>1</td><td>Shai Gilgeous-Alexander</td><td>OKC Thunder</td><td>32.7</td><td>5.5</td><td>6.4</td>
  </tr>
  <tr>
    <td>2</td><td>Giannis Antetokounmpo</td><td>Milwaukee Bucks</td><td>30.4</td><td>11.9</td><td>6.5</td>
  </tr>
  <tr>
    <td>3</td><td>Luka Doncic</td><td>LA Lakers</td><td>28.1</td><td>8.5</td><td>7.8</td>
  </tr>
  <tr>
    <td>4</td><td>Kevin Durant</td><td>Phoenix Suns</td><td>27.3</td><td>6.8</td><td>3.9</td>
  </tr>
  <tr>
    <td>5</td><td>Anthony Edwards</td><td>Minnesota Timberwolves</td><td>27.1</td><td>5.4</td><td>5.0</td>
  </tr>
</table>
</body></html>
"""

soup4 = BeautifulSoup(html4, 'html.parser')

# Step 1 — Find the table
table = soup4.find('table', id='players')
print('Found table:', table is not None)

In [ ]:
# Step 2 — Extract headers from <th> tags
headers = [th.get_text(strip=True) for th in table.find_all('th')]
print('Headers:', headers)

In [ ]:
# Step 3 — Extract data rows (skip the first row — it's the header)
rows = []
for tr in table.find_all('tr')[1:]:       # [1:] skips the <th> row
    cells = tr.find_all('td')
    row = [cell.get_text(strip=True) for cell in cells]
    if row:                                # skip any empty rows
        rows.append(row)

print('Rows extracted:', len(rows))
print('First row:', rows[0])

In [ ]:
# Step 4 — Build the DataFrame
df = pd.DataFrame(rows, columns=headers)

# Convert numeric columns
for col in ['Rank', 'PPG', 'RPG', 'APG']:
    df[col] = pd.to_numeric(df[col])

print(df)
print()
print('Data types:')
print(df.dtypes)

In [ ]:
# Now we can use all our Pandas skills!
print('--- Average PPG across top 5 ---')
print(f"{df['PPG'].mean():.1f} points per game")
print()

print('--- Player with most rebounds ---')
best_rebounder = df.loc[df['RPG'].idxmax()]
print(f"{best_rebounder['Player']} ({best_rebounder['RPG']} RPG)")
print()

print('--- Players averaging 6+ assists ---')
print(df[df['APG'] >= 6][['Player', 'APG']])

---
## Section 5 — Searching by CSS Class: Cards Pattern

Many real websites use **card layouts** — repeated `<div>` blocks with the same class.  
This is the pattern for scraping product listings, news articles, player profiles, etc.

In [ ]:
html5 = """
<html><body>
<div class="player-card">
  <h3 class="name">LeBron James</h3>
  <span class="team">Los Angeles Lakers</span>
  <span class="stat">PPG: 24.5</span>
  <a class="profile-link" href="/players/lebron">View Profile</a>
</div>
<div class="player-card">
  <h3 class="name">Stephen Curry</h3>
  <span class="team">Golden State Warriors</span>
  <span class="stat">PPG: 26.4</span>
  <a class="profile-link" href="/players/curry">View Profile</a>
</div>
<div class="player-card">
  <h3 class="name">Nikola Jokic</h3>
  <span class="team">Denver Nuggets</span>
  <span class="stat">PPG: 26.0</span>
  <a class="profile-link" href="/players/jokic">View Profile</a>
</div>
</body></html>
"""

soup5 = BeautifulSoup(html5, 'html.parser')

# Find all player cards
cards = soup5.find_all('div', class_='player-card')
print(f'Found {len(cards)} player cards\n')

# Build a list of dictionaries, then convert to DataFrame
data = []
for card in cards:
    record = {
        'Name':    card.find('h3', class_='name').get_text(strip=True),
        'Team':    card.find('span', class_='team').get_text(strip=True),
        'Stat':    card.find('span', class_='stat').get_text(strip=True),
        'Profile': card.find('a', class_='profile-link').get('href')
    }
    data.append(record)

df2 = pd.DataFrame(data)
print(df2)

---
## Section 6 — Live Web Request: Fetching a Real Page

Now we put `requests` and `BeautifulSoup` together to fetch real data from the internet.  
We'll use a simple, stable Wikipedia page as our target.

In [ ]:
import requests
from bs4 import BeautifulSoup

# Fetch a Wikipedia page
url = "https://en.wikipedia.org/wiki/List_of_most-watched_television_broadcasts_in_the_United_States"

response = requests.get(url)
print(f'Status code: {response.status_code}')   # 200 = success
print(f'Page size:   {len(response.text):,} characters')

In [ ]:
soup6 = BeautifulSoup(response.text, 'html.parser')

# Get the page title
print('Page title:', soup6.title.text)
print()

# Get the main heading
print('Main heading:', soup6.find('h1').text)
print()

# Count how many tables are on the page
tables = soup6.find_all('table')
print(f'Tables found: {len(tables)}')

In [ ]:
# Use pd.read_html as a quick shortcut for well-structured Wikipedia tables
tables_df = pd.read_html(url)
print(f'pd.read_html found {len(tables_df)} tables\n')

# Look at the first table
df_tv = tables_df[0]
print('Shape:', df_tv.shape)
print()
print(df_tv.head(10))

---
## Section 7 — Ethical Scraping with `time.sleep()`

When scraping multiple pages, always add a delay between requests.

In [ ]:
import time

# Example: scraping multiple Wikipedia pages with a polite delay
urls = [
    "https://en.wikipedia.org/wiki/LeBron_James",
    "https://en.wikipedia.org/wiki/Stephen_Curry",
    "https://en.wikipedia.org/wiki/Kevin_Durant"
]

results = []

for url in urls:
    response = requests.get(url)
    
    if response.status_code == 200:
        s = BeautifulSoup(response.text, 'html.parser')
        title = s.find('h1').get_text(strip=True)
        first_para = s.find('p').get_text(strip=True)[:120]   # first 120 chars
        results.append({'Name': title, 'Preview': first_para})
        print(f'✅ Scraped: {title}')
    else:
        print(f'❌ Failed ({response.status_code}): {url}')
    
    time.sleep(1)   # wait 1 second between requests — be polite!

print()
df_players = pd.DataFrame(results)
print(df_players[['Name']])

---
## Summary: Beautiful Soup Cheat Sheet

| Task | Code |
|---|---|
| Parse HTML string | `soup = BeautifulSoup(html, 'html.parser')` |
| Parse from URL | `soup = BeautifulSoup(requests.get(url).text, 'html.parser')` |
| Find first tag | `soup.find('p')` |
| Find all tags | `soup.find_all('a')` |
| Find by class | `soup.find('div', class_='card')` |
| Find by id | `soup.find('div', id='main')` |
| Get text | `tag.get_text(strip=True)` |
| Get attribute | `tag.get('href')` |
| Table shortcut | `pd.read_html(url)[0]` |
| Polite delay | `time.sleep(1)` |

---
## 🚀 What's Next — Week 27: APIs and JSON

Next week we learn about **web APIs** — a cleaner, faster way to get data when a site provides one.  
APIs return **JSON** (JavaScript Object Notation), which Python handles beautifully with the `json` library.

---
*Python for Data Science is offered by **Learn and Help** ([www.learnandhelp.com](https://www.learnandhelp.com)) — empowering students through technology and service.*